[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/03_baseline_rf.ipynb)

In [ ]:
import os, sys
from pathlib import Path

# ── Detectar si estamos en Google Colab ──────────────────────────────────
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

DATA_ROOT_OVERRIDE = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # Estrategia 1: acceso por ID de carpeta raíz del proyecto
    _p1 = "/content/drive/.shortcut-targets-by-id/13VEeA8Jt0G_NNpJelWZrhacBhA_mRpnU"
    # Estrategia 2: acceso por ID de TrainData (subir un nivel)
    _p2 = "/content/drive/.shortcut-targets-by-id/13Dj5DYGOwgMGomMC8gCySKGr0ne8hOpU"

    if os.path.exists(_p1) and os.path.isdir(_p1):
        DATA_ROOT_OVERRIDE = _p1
        print(f"Estrategia 1 OK: {_p1}")
    elif os.path.exists(_p2) and os.path.isdir(_p2):
        DATA_ROOT_OVERRIDE = str(Path(_p2).parent)
        print(f"Estrategia 2 OK: {DATA_ROOT_OVERRIDE}")
    else:
        import subprocess
        _r = subprocess.run(
            ["find", "/content/drive/MyDrive", "-name", "TrainData",
             "-type", "d", "-maxdepth", "6"],
            capture_output=True, text=True, timeout=30
        )
        _hits = [p.replace('/TrainData', '') for p in _r.stdout.strip().splitlines() if p]
        if _hits:
            DATA_ROOT_OVERRIDE = _hits[0]
            print(f"Estrategia 3 OK: {DATA_ROOT_OVERRIDE}")
        else:
            print("No se encontró el dataset. Ajusta DATA_ROOT_OVERRIDE manualmente:")
            print("  DATA_ROOT_OVERRIDE = 'ruta/en/tu/Drive'")

    if DATA_ROOT_OVERRIDE:
        print(f"Dataset: {DATA_ROOT_OVERRIDE}")
else:
    print('Entorno local — se usará detección automática de rutas.')


# 03 — Baseline Random Forest (HOG Features)

Establecemos el baseline clásico con Random Forest sobre características HOG, para comparar con las arquitecturas deep learning.

---

In [ ]:
import os, sys
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN DEL DATASET
# Si la celda de Drive de arriba encontró el dataset, DATA_ROOT_OVERRIDE
# ya está definida. Si no, puedes ajustarla manualmente aquí:
# DATA_ROOT_OVERRIDE = "/content/drive/MyDrive/landslide4sense"
# ══════════════════════════════════════════════════════════════════════════════

# Preservar DATA_ROOT_OVERRIDE si ya fue fijada por la celda de Drive
try:
    DATA_ROOT_OVERRIDE
except NameError:
    DATA_ROOT_OVERRIDE = None

# ── Detección automática ──────────────────────────────────────────────────────
if DATA_ROOT_OVERRIDE:
    DATA_ROOT = Path(DATA_ROOT_OVERRIDE)
    print(f"Usando ruta configurada: {DATA_ROOT}")
else:
    _cwd = Path(os.getcwd())
    _candidates = [
        _cwd / ".." / "data",
        _cwd / "..",
        _cwd / "data",
        _cwd,
        Path("/content/drive/MyDrive/landslide4sense"),
        Path("/content/drive/MyDrive/Landslide_ML"),
        Path("/content/landslide4sense"),
        Path("/content/data"),
        Path("/content"),
    ]
    DATA_ROOT = None
    for _c in _candidates:
        if (_c / "TrainData" / "img").exists():
            DATA_ROOT = _c.resolve()
            print(f"Dataset detectado automáticamente en: {DATA_ROOT}")
            break

if DATA_ROOT is None or not (DATA_ROOT / "TrainData" / "img").exists():
    print("""
⚠️  Dataset no encontrado. Opciones:

━━━  OPCIÓN A — Google Drive (recomendado)  ━━━
  Ejecuta la celda de Drive de arriba y vuelve a correr esta celda.

━━━  OPCIÓN B — Kaggle API  ━━━
  !pip install kaggle -q
  # Sube tu kaggle.json y ejecuta:
  !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
  !kaggle datasets download -d landslide4sense/competition -p /content/
  !unzip -q /content/competition.zip -d /content/landslide4sense/
  DATA_ROOT_OVERRIDE = "/content/landslide4sense"
""")
    raise FileNotFoundError("Dataset no encontrado. Establece DATA_ROOT_OVERRIDE con la ruta correcta.")

# ── Verificar estructura del dataset ─────────────────────────────────────────
n_train = len(list((DATA_ROOT / "TrainData" / "img").glob("*.h5")))
print(f"✓ Dataset encontrado en: {DATA_ROOT}")
print(f"  TrainData/img : {n_train} archivos .h5")


## 1. Extracción de características HOG

In [ ]:
img_dir  = DATA_ROOT / "TrainData" / "img"
mask_dir = DATA_ROOT / "TrainData" / "mask"
files    = sorted(img_dir.glob("*.h5"))[:N_SAMPLES]

X, y = [], []
for f in tqdm(files, desc="Extrayendo HOG"):
    mf = mask_dir / f.name
    with h5py.File(f,  "r") as hf: patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mf, "r") as hf: mask  = hf[list(hf.keys())[0]][()]
    rgb = patch[:,:,[2,1,0]]
    rgb = ((rgb - rgb.min()) / (rgb.ptp() + 1e-8) * 255).astype("uint8")
    feats = hog(rgb, pixels_per_cell=(16,16), cells_per_block=(2,2), channel_axis=-1)
    X.append(feats)
    y.append(int(mask.max() > 0))

X, y = np.array(X), np.array(y)
print(f"Features HOG: {X.shape}")
print(f"Positivos: {y.sum()} ({100*y.mean():.1f}%)  |  Negativos: {(1-y).sum()} ({100*(1-y).mean():.1f}%)")

## 2. Entrenamiento K-Fold (5-Fold)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1s, aucs, precisions, recalls = [], [], [], []

for fold, (tr, va) in enumerate(skf.split(X, y)):
    rf = RandomForestClassifier(n_estimators=200, max_depth=None, n_jobs=-1, random_state=42)
    rf.fit(X[tr], y[tr])
    probs = rf.predict_proba(X[va])[:,1]
    preds = (probs >= 0.5).astype(int)
    
    f1  = f1_score(y[va], preds, zero_division=0)
    auc = roc_auc_score(y[va], probs)
    from sklearn.metrics import precision_score, recall_score
    pre = precision_score(y[va], preds, zero_division=0)
    rec = recall_score(y[va], preds, zero_division=0)
    
    f1s.append(f1); aucs.append(auc)
    precisions.append(pre); recalls.append(rec)
    print(f"  Fold {fold}: F1={f1:.4f}  AUC={auc:.4f}  Prec={pre:.4f}  Rec={rec:.4f}")

print(f"\nResumen 5-Fold CV:")
print(f"  F1-score:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"  AUC-ROC:   {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"  Precision: {np.mean(precisions):.4f} ± {np.std(precisions):.4f}")
print(f"  Recall:    {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")

## 3. Importancia de características

In [ ]:
# Entrenar RF final sobre todos los datos para visualizar feature importance
rf_final = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
rf_final.fit(X, y)
importances = rf_final.feature_importances_

# Agrupar por bloque de características
top_k = 20
top_idx = np.argsort(importances)[-top_k:][::-1]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(top_k), importances[top_idx], color="#3498db", alpha=0.85)
ax.set_xticks(range(top_k))
ax.set_xticklabels([f"F#{i}" for i in top_idx], rotation=45, fontsize=9)
ax.set_ylabel("Importancia (Gini)")
ax.set_title(f"Top {top_k} características HOG más discriminativas — Random Forest", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Suma top-{top_k}: {importances[top_idx].sum():.3f} ({100*importances[top_idx].sum():.1f}% de la importancia total)")

## 4. Guardar resultados baseline

In [ ]:
import json
results = {
    "model": "RandomForest_HOG", "n_samples": N_SAMPLES,
    "n_estimators": 200, "n_folds": 5,
    "f1_mean":  float(np.mean(f1s)),  "f1_std":  float(np.std(f1s)),
    "auc_mean": float(np.mean(aucs)), "auc_std": float(np.std(aucs)),
    "precision_mean": float(np.mean(precisions)), "recall_mean": float(np.mean(recalls)),
}
Path("../results/random_forest").mkdir(parents=True, exist_ok=True)
with open("../results/random_forest/rf_summary.json","w") as f:
    json.dump(results, f, indent=2)
print("Resultados guardados en results/random_forest/rf_summary.json")
print(f"\nBaseline establecido: F1 = {results['f1_mean']:.4f} ± {results['f1_std']:.4f}")